In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from urllib.request import urlopen
from importlib.metadata import version
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, LlamaConfig

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
settings = json.loads((TASK / 'input/config.json').read_text())
cfg = {k: value for k, value in settings.items() if k != 'profiles'}
cfg.update(settings['profiles'][PROFILE])
cfg['profile'] = PROFILE
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])
assert SEED == 63 and cfg['sources'] == 2
assert cfg['backend'] in ['pretrained', 'random_fixture']
assert cfg['finetune_steps'][0] == 0 and cfg['finetune_steps'] == sorted(set(cfg['finetune_steps']))
assert cfg['references_per_source'] >= 1 and min(cfg['marker_counts']) >= 1
assert all(n > 0 for n in [cfg['sequence_length'], cfg['family_steps'], cfg['child_steps'], cfg['evaluation_batches']])
assert cfg['sequence_length'] > 1 and 0 <= min(cfg['lambda_grid'])

In [ ]:
for cell in json.loads((TASK / 'src/helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
cache = TASK.parent / '.cache'
cache.mkdir(exist_ok=True)
if cfg['backend'] == 'pretrained':
    model_path = Path(snapshot_download(cfg['model_id'], revision=cfg['revision'], cache_dir=str(cache / 'hub'),
        allow_patterns=['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json',
                        'special_tokens_map.json', 'vocab.json', 'merges.txt'], max_workers=2, token=False))
    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    corpus_file = cache / 'tiny_shakespeare.txt'
    if not corpus_file.is_file():
        with urlopen(cfg['corpus_url'], timeout=60) as response:
            corpus_file.write_bytes(response.read())
    raw = corpus_file.read_bytes()
    text = raw.decode('utf-8')[:cfg['max_characters']]
    tokens = np.array(tokenizer(text, add_special_tokens=False, verbose=False)['input_ids'], dtype=np.int64)
    source = {'corpus_sha256': hashlib.sha256(raw).hexdigest(), 'tokenizer_revision': cfg['revision']}
else:
    # explicit offline fixture; never substituted for pretrained results
    model_path = cache / 'random_fixture'
    model_path.mkdir(exist_ok=True)
    model = AutoModelForCausalLM.from_config(LlamaConfig(vocab_size=256, hidden_size=32, intermediate_size=64,
        num_hidden_layers=6, num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=128,
        tie_word_embeddings=True, attention_dropout=0.0, bos_token_id=0, eos_token_id=1))
    model.save_pretrained(str(model_path), safe_serialization=True)
    del model
    tokens = np.r_[rng.integers(2, 128, size=2000), rng.integers(128, 256, size=2000), rng.integers(2, 256, size=1000)]
    source = {'corpus_sha256': None, 'tokenizer_revision': None}
model_config = AutoConfig.from_pretrained(str(model_path), local_files_only=True)
assert model_config.model_type == 'llama' and model_config.num_hidden_layers >= 4
assert (model_path / 'model.safetensors').is_file()
assert max(tokens) < model_config.vocab_size

In [ ]:
split = int(0.8 * len(tokens))
training, evaluation = tokens[:split], tokens[split:]
family0, family1 = np.array_split(training, 2)
assert min(len(family0), len(family1), len(evaluation)) > cfg['sequence_length']
starts = rng.integers(0, len(evaluation) - cfg['sequence_length'] + 1, size=cfg['evaluation_batches'])
evaluation_ids = np.stack([evaluation[s:s + cfg['sequence_length']] for s in starts])
np.savez_compressed(OUTPUT / 'tokens.npz', training=training, family0=family0, family1=family1, evaluation_ids=evaluation_ids)
(OUTPUT / 'model').unlink(missing_ok=True)
(OUTPUT / 'model').symlink_to(os.path.relpath(model_path, OUTPUT), target_is_directory=True)
(OUTPUT / 'config.json').write_text(json.dumps(cfg, indent=2))
shutil.copy2(TASK / 'src/helpers.ipynb', OUTPUT / 'helpers.ipynb')
environment = {'profile': PROFILE, 'backend': cfg['backend'], 'model_id': cfg['model_id'],
    'revision': cfg['revision'] if cfg['backend'] == 'pretrained' else None, 'seed': SEED,
    'blocks': model_config.num_hidden_layers, **source,
    'packages': {name: version(name) for name in ['torch', 'transformers', 'numpy', 'pandas', 'scipy', 'safetensors']}}
(OUTPUT / 'environment.json').write_text(json.dumps(environment, indent=2))
display(pd.DataFrame({'profile': [PROFILE], 'backend': [cfg['backend']], 'blocks': [model_config.num_hidden_layers],
    'training_tokens': [len(training)], 'evaluation_tokens': [len(evaluation)]}))